# Optimization Pipeline for JobStreet Dataset

This notebook demonstrates three optimization techniques applied to the cleaned JobStreet dataset:

1. Pandas Optimized Pipeline
2. Polars Lazy Execution Pipeline
3. DuckDB SQL-Based Processing

The objective is to improve data processing efficiency when handling large-scale job listing datasets.

### Install and Import Libraries

In [1]:
!pip install polars duckdb
import pandas as pd
import polars as pl
import duckdb

###Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
CSV_FILE = "/content/drive/MyDrive/HPDP/Project 1/cleaned_data.csv"
OUTPUT_FOLDER = "/content/drive/MyDrive/HPDP/Project 1/"
required_cols = ["job_title", "company", "location", "classification", "salary"]

# 1. Pandas Optimized Pipeline
## Pandas Optimized Pipeline

The Pandas pipeline improves processing efficiency through:

- usecols
- dtype specification
- category data types
- vectorized string operations
- optimized groupby aggregation

In [4]:
required_cols = [
    "job_title",
    "company",
    "location",
    "classification",
    "salary"
]

df_pandas = pd.read_csv(
    CSV_FILE,
    usecols=required_cols,
    dtype={
        "job_title": "string",
        "company": "string",
        "location": "string",
        "classification": "string",
        "salary": "string"
    },
    engine="c"
)

df_pandas.head()

,job_title,company,location,classification,salary
0,EOI- Facilities Manager_Soft service (Penang),CBRE,George Town,Real Estate & Property,Not Disclosed
1,Senior Electrical Engineer- EHV Cables,AECOM,Damansara,Engineering,Not Disclosed
2,Management Consultant - ERP Platforms (SAP FI-CO),Accenture Malaysia,Kuala Lumpur City Centre,Information & Communication Technology,Not Disclosed
3,Accountant,Nala Groups on behalf of Nala Groups,Johor Bahru,Accounting,Not Disclosed
4,Marketing Executive,Nalagroups,Gelang Patah,Marketing & Communications,Not Disclosed


In [5]:
df_pandas["location"] = df_pandas["location"].astype("category")
df_pandas["classification"] = df_pandas["classification"].astype("category")

In [6]:
df_pandas["job_title"] = (
    df_pandas["job_title"]
    .str.strip()
    .str.title()
)

df_pandas["company"] = (
    df_pandas["company"]
    .str.strip()
    .str.title()
)

In [7]:
mask_it = df_pandas["classification"].astype(str).str.contains(
    "Information & Communication Technology",
    case=False,
    na=False
)

df_it_jobs = df_pandas.loc[mask_it]

df_it_jobs.head()

,job_title,company,location,classification,salary
2,Management Consultant - Erp Platforms (Sap Fi-Co),Accenture Malaysia,Kuala Lumpur City Centre,Information & Communication Technology,Not Disclosed
6,Mobile Architect,Luxoft Malaysia Sdn. Bhd.,Malaysia,Information & Communication Technology,Not Disclosed
7,Murex Technical Business Analyst,Luxoft Malaysia Sdn. Bhd.,Kuala Lumpur City Centre,Information & Communication Technology,Not Disclosed
9,Java Developer,Dxc Technology Malaysia Sdn Bhd.,Petaling,Information & Communication Technology,Not Disclosed
16,Senior Servicenow Developer,Swift Support Services Malaysia Sdn. Bhd.,Kuala Lumpur,Information & Communication Technology,Not Disclosed


In [8]:
pandas_summary = (
    df_pandas
    .groupby("classification", observed=True)
    .agg(
        total_jobs=("job_title", "count"),
        total_companies=("company", "nunique")
    )
    .reset_index()
    .sort_values("total_jobs", ascending=False)
)

pandas_summary.head(10)

,classification,total_jobs,total_companies
20,"Manufacturing, Transport & Logistics",3145,1774
0,Accounting,2935,2077
11,Engineering,2747,1458
25,Sales,2708,1755
17,Information & Communication Technology,2375,1100
1,Administration & Office Support,1657,1341
21,Marketing & Communications,1402,1015
16,Human Resources & Recruitment,970,757
24,Retail & Consumer Products,947,465
7,Construction,930,582


#2. Polars
## Polars Lazy Execution

Polars improves performance through:

- scan_csv()
- lazy evaluation
- expression API
- parallel execution
- collect() only when required

In [9]:
lf = pl.scan_csv(CSV_FILE)

lf_cleaned = lf.select([
    pl.col("job_title").str.strip_chars().str.to_titlecase(),
    pl.col("company").str.strip_chars().str.to_titlecase(),
    pl.col("location").str.strip_chars(),
    pl.col("classification").str.strip_chars(),
    pl.col("salary")
])

In [10]:
lf_it_jobs = lf_cleaned.filter(
    pl.col("classification")
    .str.contains(
        "Information & Communication Technology",
        literal=True
    )
)

lf_it_jobs.collect().head()

job_title,company,location,classification,salary
str,str,str,str,str
"""Management Consultant - Erp Pl…","""Accenture Malaysia""","""Kuala Lumpur City Centre""","""Information & Communication Te…","""Not Disclosed"""
"""Mobile Architect""","""Luxoft Malaysia Sdn. Bhd.""","""Malaysia""","""Information & Communication Te…","""Not Disclosed"""
"""Murex Technical Business Analy…","""Luxoft Malaysia Sdn. Bhd.""","""Kuala Lumpur City Centre""","""Information & Communication Te…","""Not Disclosed"""
"""Java Developer""","""Dxc Technology Malaysia Sdn Bh…","""Petaling""","""Information & Communication Te…","""Not Disclosed"""
"""Senior Servicenow Developer""","""Swift Support Services Malaysi…","""Kuala Lumpur""","""Information & Communication Te…","""Not Disclosed"""


In [11]:
polars_summary = (
    lf_cleaned
    .group_by("classification")
    .agg([
        pl.count().alias("total_jobs"),
        pl.col("company").n_unique().alias("total_companies")
    ])
    .sort("total_jobs", descending=True)
    .collect()
)

polars_summary.head(10)

/tmp/ipykernel_2124/837103016.py:5: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("total_jobs"),


classification,total_jobs,total_companies
str,u32,u32
"""Manufacturing, Transport & Log…",3145,1774
"""Accounting""",2935,2077
"""Engineering""",2747,1458
"""Sales""",2708,1755
"""Information & Communication Te…",2375,1100
"""Administration & Office Suppor…",1657,1341
"""Marketing & Communications""",1402,1015
"""Human Resources & Recruitment""",970,757
"""Retail & Consumer Products""",947,465


# 3. DuckDB
## DuckDB Analytical Processing

DuckDB processes CSV data directly using SQL queries without fully loading the dataset into memory.

Optimization techniques:

- read_csv_auto()
- SQL aggregation
- GROUP BY optimization
- REGEXP filtering

In [12]:
con = duckdb.connect()

In [13]:
duckdb_summary = con.execute(f"""
SELECT
    classification,
    COUNT(*) AS total_jobs,
    COUNT(DISTINCT company) AS total_companies
FROM read_csv_auto('{CSV_FILE}')
GROUP BY classification
ORDER BY total_jobs DESC
""").df()

duckdb_summary.head(10)

,classification,total_jobs,total_companies
0,"Manufacturing, Transport & Logistics",3145,1776
1,Accounting,2935,2082
2,Engineering,2747,1460
3,Sales,2708,1755
4,Information & Communication Technology,2375,1100
5,Administration & Office Support,1657,1342
6,Marketing & Communications,1402,1016
7,Human Resources & Recruitment,970,757
8,Retail & Consumer Products,947,467
9,Construction,930,584


In [14]:
duckdb_it_jobs = con.execute(f"""
SELECT *
FROM read_csv_auto('{CSV_FILE}')
WHERE REGEXP_MATCHES(
    classification,
    'Information & Communication Technology',
    'i'
)
""").df()

duckdb_it_jobs.head()

,job_title,company,location,classification,salary,salary_min,salary_max
0,Management Consultant - ERP Platforms (SAP FI-CO),Accenture Malaysia,Kuala Lumpur City Centre,Information & Communication Technology,Not Disclosed,<NA>,<NA>
1,Mobile Architect,LUXOFT MALAYSIA SDN. BHD.,Malaysia,Information & Communication Technology,Not Disclosed,<NA>,<NA>
2,Murex Technical Business Analyst,LUXOFT MALAYSIA SDN. BHD.,Kuala Lumpur City Centre,Information & Communication Technology,Not Disclosed,<NA>,<NA>
3,Java Developer,DXC Technology Malaysia Sdn Bhd.,Petaling,Information & Communication Technology,Not Disclosed,<NA>,<NA>
4,Senior ServiceNow Developer,SWIFT SUPPORT SERVICES MALAYSIA SDN. BHD.,Kuala Lumpur,Information & Communication Technology,Not Disclosed,<NA>,<NA>


In [15]:
con.close()